In [ ]:
import casadi as ca

x = ca.SX.sym('x')
y = ca.SX.sym('y')

f = x**2 + y**2 + 2*x*y

F = ca.Function('F', [x, y], [f]) # 최적화 변수?, 목적함수

In [ ]:
print(F(2,3))

In [ ]:

x = ca.SX.sym('x', 3)  # 3차원 벡터
f = ca.vertcat(
    x[0]**2 + x[1],
    ca.sin(x[2]) + x[0]*x[1]
)

# 야코비안
J = ca.jacobian(f, x)
J_func = ca.Function('J', [x], [J])
print(J_func([1, 2, 0.5]))

# 그래디언트와 헤시안
g = x[0]**2 + x[1]**2 + x[2]**2
grad = ca.gradient(g, x)
hess = ca.hessian(g, x)[0]

In [ ]:
import casadi as ca

# 변수
x = ca.SX.sym('x')
y = ca.SX.sym('y')

# 목적 함수 (Rosenbrock)
f = (1 - x)**2 + 100*(y - x**2)**2

# 제약조건: x^2 + y^2 <= 1
g = x**2 + y**2

# NLP 정의
nlp = {'x': ca.vertcat(x, y), 'f': f, 'g': g}

# Solver 생성 (IPOPT)
solver = ca.nlpsol('solver', 'ipopt', nlp)

# 풀기
sol = solver(x0=[0, 0], lbg=-ca.inf, ubg=1.0)
print('최적해:', sol['x'])
print('최적값:', sol['f'])

In [ ]:
import casadi as ca

N = 20      # 예측 구간
T = 2.0     # 총 시간
dt = T / N

# 상태와 제어 변수
x = ca.SX.sym('x', 2)  # [위치, 속도]
u = ca.SX.sym('u')      # 힘

# 동역학 (간단한 1차원 시스템)
xdot = ca.vertcat(x[1], u - 0.1*x[1])
f = ca.Function('f', [x, u], [xdot])

# RK4 적분기
def rk4(x, u, dt):
    k1 = f(x, u)
    k2 = f(x + dt/2 * k1, u)
    k3 = f(x + dt/2 * k2, u)
    k4 = f(x + dt * k3, u)
    return x + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

# 최적화 변수
opti = ca.Opti()
X = opti.variable(2, N+1)
U = opti.variable(1, N)

# 목적 함수: 목표점 [1, 0]에 도달 + 제어 노력 최소화
cost = 0
for k in range(N):
    cost += (X[0,k] - 1)**2 + X[1,k]**2 + 0.01*U[0,k]**2
    opti.subject_to(X[:,k+1] == rk4(X[:,k], U[0,k], dt))

opti.minimize(cost)

# 초기 조건과 제어 제약
opti.subject_to(X[:,0] == [0, 0])
opti.subject_to(opti.bounded(-2, U, 2))

# 풀기
opti.solver('ipopt')
sol = opti.solve()

print('최적 상태 궤적:', sol.value(X))
print('최적 제어 입력:', sol.value(U))

In [ ]:
import casadi as ca
import numpy as np

# 변수
x = ca.SX.sym('x', 2)

# 목적함수 (2차식이면 뭐든 OK)
f = x[0]**2 + x[1]**2 + x[0]*x[1] - x[0] - x[1]

# 제약식 (선형)
g_expr = ca.vertcat(
    x[0] + x[1],
    x[0],
    x[1]
)

# H = ∇²f  (Hessian)
H = ca.hessian(f, x)[0]   # [0]이 헤시안, [1]이 그래디언트

# g = ∇f |_{x=0}  (선형항 계수)
grad = ca.gradient(f, x)
g = ca.substitute(grad, x, ca.DM.zeros(x.shape))

# A = ∂g/∂x  (제약식 자코비안)
A = ca.jacobian(g_expr, x)

# numpy로 변환
H_np = np.array(ca.DM(H))
g_np = np.array(ca.DM(g)).flatten()
A_np = np.array(ca.DM(A))

print('H =\n', H_np)
print('g =', g_np)
print('A =\n', A_np)

In [ ]:
import casadi as ca
import numpy as np

# 시스템
A_sys = ca.DM([[1.0, 0.1], [0.0, 1.0]])
B_sys = ca.DM([[0.0], [0.1]])
N, nx, nu = 10, 2, 1

# 의사결정 변수
X = ca.SX.sym('X', nx, N+1)
U = ca.SX.sym('U', nu, N)
z = ca.vertcat(ca.reshape(X, -1, 1), ca.reshape(U, -1, 1))

# 파라미터: 초기 상태 (매 스텝 바뀜)
x0 = ca.SX.sym('x0', nx)

# 목적함수
Q = ca.DM.eye(nx)
R = ca.DM.eye(nu) * 0.1
cost = 0
for k in range(N):
    cost += X[:,k].T @ Q @ X[:,k] + U[:,k].T @ R @ U[:,k]
cost += X[:,N].T @ Q @ X[:,N]

# 제약식 (등식: 동역학 + 초기조건)
g_list = [X[:,0] - x0]
for k in range(N):
    g_list.append(X[:,k+1] - (A_sys @ X[:,k] + B_sys @ U[:,k]))
g_expr = ca.vertcat(*g_list)

# QP 행렬 추출
H = ca.hessian(cost, z)[0]
grad_z = ca.gradient(cost, z)
g_lin = ca.substitute(grad_z, z, ca.DM.zeros(z.shape))  # 보통 0
A_con = ca.jacobian(g_expr, z)

# x0에 의존하는 부분 (b 벡터)
b = -ca.substitute(g_expr, z, ca.DM.zeros(z.shape))  # = [x0; 0; 0; ...]

# Function으로 만들어두면 매 MPC 스텝마다 빠르게 평가 가능
get_qp = ca.Function('get_qp', [x0], [H, g_lin, A_con, b])

# 실제 사용
x0_val = ca.DM([1.0, 0.0])
H_val, g_val, A_val, b_val = get_qp(x0_val)

H_np = np.array(H_val)
g_np = np.array(g_val).flatten()
A_np = np.array(A_val)
l_np = np.array(b_val).flatten()  # 등식이므로 l = u = b
u_np = l_np.copy()